<a href="https://colab.research.google.com/github/edusatyaki/SummerDrive/blob/main/Cohort-1/E-Commerce/Data%20Cleaning/Ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

**Step 1**

We will Import pandas and read all 9 Olist CSV files into separate DataFrames — customers, orders, order_items, payments, reviews, products, sellers, geolocation, and category translation.

The dataset is split across 9 files by design — each file represents one entity (like a database table). We need all of them in memory simultaneously to be able to join and analyze them together.

No changes to data yet — just creates 9 named DataFrames. Sets up the foundation for every task that follows. Without this step, no downstream processing is possible.


In [ ]:
drive.mount('/content/drive')

In [ ]:
customers = pd.read_csv('/content/drive/MyDrive/Datasets/olist_customers_dataset.csv')
order_items = pd.read_csv('/content/drive/MyDrive/Datasets/olist_order_items_dataset.csv')
orders = pd.read_csv('/content/drive/MyDrive/Datasets/olist_orders_dataset.csv')
order_payment = pd.read_csv('/content/drive/MyDrive/Datasets/olist_order_payments_dataset.csv')
order_review = pd.read_csv('/content/drive/MyDrive/Datasets/olist_order_reviews_dataset.csv')
products = pd.read_csv('/content/drive/MyDrive/Datasets/olist_products_dataset.csv')
sellers = pd.read_csv('/content/drive/MyDrive/Datasets/olist_sellers_dataset.csv')
geolocation = pd.read_csv('/content/drive/MyDrive/Datasets/olist_geolocation_dataset.csv')
product_category_name = pd.read_csv('/content/drive/MyDrive/Datasets/product_category_name_translation.csv')

In [ ]:
customers.head()

In [ ]:
customers.info()

In [ ]:
customers.describe()

In [ ]:
customers = customers.drop_duplicates()

In [ ]:
customers['customer_city'] = customers['customer_city'].str.lower().str.strip()

In [ ]:
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)

**Step 2**

We have to Check whether every foreign key in child tables (like order_id in order_items) actually exists in the parent table (orders). Counted "orphan" records — IDs that appear in one table but have no matching record in the other.

If you merge two tables with broken FK relationships, you either silently lose rows (inner join) or get NaN-filled ghost rows (left join). Both corrupt your analysis without any error — you'd never know. Validation catches this before it happens.

No rows are deleted or modified. This is purely a quality audit. If orphans are found, we decide whether to drop them or investigate. Olist's clean dataset returns 0 for all checks — confirming all joins are safe.

In [ ]:
orphan_items = ~order_items['order_id'].isin(orders['order_id'])
print("Orphan order_items:", orphan_items.sum())

In [ ]:
orphan_payments = ~order_payment['order_id'].isin(orders['order_id'])
print("Orphan payments:", orphan_payments.sum())

In [ ]:
orphan_reviews = ~order_review['order_id'].isin(orders['order_id'])
print("Orphan reviews:", orphan_reviews.sum())

In [ ]:
orphan_orders = ~orders['customer_id'].isin(customers['customer_id'])
print("Orphan orders:", orphan_orders.sum())

In [ ]:
orphan_sellers = ~order_items['seller_id'].isin(sellers['seller_id'])
print("Orphan sellers:", orphan_sellers.sum())

In [ ]:
orphan_products = ~order_items['product_id'].isin(products['product_id'])
print("Orphan products:", orphan_products.sum())

In [ ]:
print("\nAll relationship checks done!")

**Step 3**

We must Convert the 8 timestamp columns (across orders, order_items, and order_reviews) from plain text strings like "2017-09-13 08:59:02" into proper pandas datetime objects using pd.to_datetime().

When dates are stored as strings, you cannot subtract them, extract month/year, calculate duration, or sort them correctly. Python treats "2018-01-01" as just a string — it has no concept that it's a date. Converting unlocks time-based math.

Dtype changes from object (string) to datetime64. Enables date subtraction in Task 6 (delivery_days, delay_days). Without this step, Task 6 would throw a TypeError. Also allows groupby by month/year in EDA.

In [ ]:
datetime_cols_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in datetime_cols_orders:
    orders[col] = pd.to_datetime(orders[col])

order_review['review_creation_date'] = pd.to_datetime(order_review['review_creation_date'])
order_review['review_answer_timestamp'] = pd.to_datetime(order_review['review_answer_timestamp'])

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

print("Dtypes after parsing:")
print(orders[datetime_cols_orders].dtypes)

**Step 4**

Used groupby('order_id').agg() to collapse multiple payment rows per order into one row — summing the total payment value, taking the max installments, and joining all payment types used into a single comma-separated string.

Some orders are paid with multiple methods (e.g. credit card + voucher). The raw table has one row per payment method. If merged directly, one order becomes two rows in the master table — doubling its revenue and count, silently corrupting all analysis.

Reduces payment table from ~103,000 rows to ~99,000 rows (one per order). Creates 3 new enriched columns. Ensures 1-to-1 merge with orders table. Revenue figures, order counts, and payment analysis all become accurate.

In [ ]:
payments_agg = order_payment.groupby('order_id').agg(
    total_payment_value = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_types_used  = ('payment_type', lambda x: ','.join(sorted(x.unique())))
).reset_index()

print("Original payment rows:", len(order_payment))
print("After aggregation (one per order):", len(payments_agg))
print(payments_agg.head())

**Step 5**

We should perform a series of left merges starting from the orders table — joining customers (on customer_id), payments_agg (on order_id), aggregated order_items (on order_id), and review scores (on order_id) to build one wide master DataFrame.

Analysis questions span multiple tables — "Which state has the most late deliveries?" needs orders + customers + delivery dates. Querying across 9 separate DataFrames every time is tedious. One master table lets you answer any question in a single line.

Creates a new master DataFrame with ~99,000 rows and 20+ columns combining all key fields. Individual tables remain untouched. This becomes the single source of truth for all Phase 2 EDA and visualization tasks.

In [ ]:

master = orders.merge(customers, on='customer_id', how='left')
master = master.merge(payments_agg, on='order_id', how='left')
master = master.merge(
    order_items.groupby('order_id').agg(
        total_items       = ('order_item_id', 'count'),
        total_freight     = ('freight_value', 'sum'),
        total_price       = ('price', 'sum')
    ).reset_index(),
    on='order_id', how='left'
)
master = master.merge(
    order_review[['order_id','review_score']],
    on='order_id', how='left'
)

print("Master table shape:", master.shape)
print(master.head())

**Step 6**

We will now Derive 3 new columns: delivery_days (actual days from purchase to delivery), delay_days (actual delivery minus estimated — negative means early), and is_late (a True/False flag for orders delivered after the promised date).

The raw data only has raw timestamps — it doesn't tell you if an order was fast or slow, early or late. These derived columns translate raw dates into business-meaningful metrics. is_late is especially powerful for segmentation and root cause analysis.

Adds 3 new columns to the master table. delivery_days and delay_days are integers (can be negative for early delivery). is_late is boolean. Enables questions like "late delivery rate by state" or "avg delivery time by product category" in Phase 2.

In [ ]:
master['delivery_days'] = (
    master['order_delivered_customer_date'] - master['order_purchase_timestamp']
).dt.days


master['delay_days'] = (
    master['order_delivered_customer_date'] - master['order_estimated_delivery_date']
).dt.days

master['is_late'] = master['delay_days'] > 0

print("Sample engineered features:")
print(master[['order_id','delivery_days','delay_days','is_late']].head(10))
print("\nLate delivery rate:", master['is_late'].mean().round(3))